# Car Brand Classification - Model Training & Evaluation
## Transfer Learning mit GPU
- 35 Marken
- 25500 Bilder
- ResNet50 (VGG16 alternative möglich)

In [1]:
# Imports
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, precision_score, recall_score, f1_score
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import ResNet50, VGG16, EfficientNetB0
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
import warnings
warnings.filterwarnings('ignore')

print(f"TensorFlow Version: {tf.__version__}")
print(f"GPU Available: {tf.config.list_physical_devices('GPU')}")

TensorFlow Version: 2.21.0
GPU Available: []


In [ ]:
# ===== 1. DATASET LADEN =====
base_dir = r"C:\Users\pparr\Documents\Henallux\Henallux PP 2025_2026\Semester 2\Systèmes_intelligents"
csv_path = os.path.join(base_dir, r"Deep_learning_projet_SN_PP\combined_dataset.csv")

# CSV laden
df = pd.read_csv(csv_path)

# Pfade reparieren (falls nötig)
def fix_path(relative_path):
    """Konvertiert relative Pfade zu absoluten"""
    full_path = os.path.join(base_dir, "Dataset", relative_path)
    return full_path

df['path'] = df['path'].apply(fix_path)

# Überprüfung
print(f"Gesamte Datensätze: {len(df)}")
print(f"Anzahl Marken: {df['brand'].nunique()}")
print(f"\nBilder pro Marke:")
print(df['brand'].value_counts())

# Test: erste 3 Bilder auf Existenz checken
print(f"\nTest ob Pfade existieren:")
for idx in range(min(3, len(df))):
    path = df.iloc[idx]['path']
    exists = os.path.exists(path)
    print(f"{'✓' if exists else '✗'} {path[:80]}...")

In [ ]:
# ===== 2. TRAIN/VALIDATION/TEST SPLIT =====
# 70% Train, 15% Validation, 15% Test

# Encode labels
unique_brands = df['brand'].unique()
brand_to_idx = {brand: idx for idx, brand in enumerate(unique_brands)}
idx_to_brand = {idx: brand for brand, idx in brand_to_idx.items()}

df['label'] = df['brand'].map(brand_to_idx)

# Split
train_df, temp_df = train_test_split(df, test_size=0.30, random_state=42, stratify=df['brand'])
val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=42, stratify=temp_df['brand'])

print(f"Training Set: {len(train_df)} Bilder ({len(train_df)/len(df)*100:.1f}%)")
print(f"Validation Set: {len(val_df)} Bilder ({len(val_df)/len(df)*100:.1f}%)")
print(f"Test Set: {len(test_df)} Bilder ({len(test_df)/len(df)*100:.1f}%)")

In [ ]:
# ===== 3. DATA AUGMENTATION & IMAGE LOADING =====
IMAGE_SIZE = (224, 224)  # ResNet Input Size
BATCH_SIZE = 32
NUM_CLASSES = len(unique_brands)

# Data Augmentation (nur für Training)
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

# Nur Rescaling für Val/Test
val_datagen = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)

# Generatoren
train_generator = train_datagen.flow_from_dataframe(
    train_df,
    x_col='path',
    y_col='brand',
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True
)

val_generator = val_datagen.flow_from_dataframe(
    val_df,
    x_col='path',
    y_col='brand',
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

test_generator = test_datagen.flow_from_dataframe(
    test_df,
    x_col='path',
    y_col='brand',
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

print(f"Training Generator erstellt: {len(train_generator)} Batches")
print(f"Validation Generator erstellt: {len(val_generator)} Batches")
print(f"Test Generator erstellt: {len(test_generator)} Batches")

In [ ]:
# ===== 4. MODELL ERSTELLEN (TRANSFER LEARNING) =====
# ResNet50 mit vortrainierten weights von ImageNet

# Base Model
base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False  # Freeze weights

# Custom Top Layer
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(512, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(NUM_CLASSES, activation='softmax')
])

# Compile
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("Modell Architektur:")
model.summary()

In [ ]:
# ===== 5. CALLBACKS FÜR TRAINING =====
callbacks = [
    # Early Stopping - stoppt wenn Validation Loss nicht besser wird
    EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    
    # Speichert bestes Modell
    ModelCheckpoint(
        os.path.join(base_dir, r"Deep_learning_projet_SN_PP\best_model.h5"),
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    ),
    
    # Reduziert Learning Rate wenn Validation Loss nicht besser wird
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-7,
        verbose=1
    )
]

print("Callbacks konfiguriert!")

In [ ]:
# ===== 6. TRAINING =====
# WARNUNG: Das kann mehrere Stunden dauern! (Abhängig von GPU)

EPOCHS = 50  # Mit Early Stopping stoppt es früher wenn nötig

history = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=val_generator,
    callbacks=callbacks,
    verbose=1
)

print("\n✓ Training abgeschlossen!")

In [ ]:
# ===== 7. TRAINING HISTORY VISUALISIERUNG =====
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Accuracy
ax1.plot(history.history['accuracy'], label='Training Accuracy')
ax1.plot(history.history['val_accuracy'], label='Validation Accuracy')
ax1.set_title('Model Accuracy')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend()
ax1.grid(True)

# Loss
ax2.plot(history.history['loss'], label='Training Loss')
ax2.plot(history.history['val_loss'], label='Validation Loss')
ax2.set_title('Model Loss')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.savefig(os.path.join(base_dir, r"Deep_learning_projet_SN_PP\training_history.png"), dpi=100)
plt.show()

print("✓ Training History gespeichert")

In [ ]:
# ===== 8. EVALUATION AUF TEST SET =====
# Predictions
y_pred_probs = model.predict(test_generator, verbose=1)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = test_generator.classes

# Metrics
accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, average='weighted', zero_division=0)
recall = recall_score(y_true, y_pred, average='weighted', zero_division=0)
f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)

print(f"\n===== TEST SET EVALUATION =====")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")
print(f"==============================")

In [ ]:
# ===== 9. CLASSIFICATION REPORT =====
brand_names = [idx_to_brand[i] for i in range(NUM_CLASSES)]

report = classification_report(
    y_true, y_pred,
    target_names=brand_names,
    zero_division=0
)

print("\nDetaillierter Classification Report:")
print(report)

# Speichern
report_path = os.path.join(base_dir, r"Deep_learning_projet_SN_PP\classification_report.txt")
with open(report_path, 'w') as f:
    f.write(report)
print(f"\n✓ Gespeichert: {report_path}")

In [ ]:
# ===== 10. CONFUSION MATRIX =====
cm = confusion_matrix(y_true, y_pred)

# Große Confusion Matrix
plt.figure(figsize=(20, 18))
sns.heatmap(cm, annot=False, fmt='d', cmap='Blues', xticklabels=brand_names, yticklabels=brand_names)
plt.title('Confusion Matrix - All Brands')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig(os.path.join(base_dir, r"Deep_learning_projet_SN_PP\confusion_matrix_full.png"), dpi=100)
plt.show()
print("✓ Confusion Matrix gespeichert")

In [ ]:
# ===== 11. TOP FEHLER ANZEIGEN =====
# Zeige die Marken mit den meisten Fehlern

errors_per_brand = {}
for i in range(NUM_CLASSES):
    total = np.sum(y_true == i)
    correct = np.sum((y_true == i) & (y_pred == i))
    errors = total - correct
    errors_per_brand[brand_names[i]] = {'total': total, 'errors': errors, 'accuracy': correct/total*100 if total > 0 else 0}

# Sortiert nach Fehler
errors_sorted = sorted(errors_per_brand.items(), key=lambda x: x[1]['errors'], reverse=True)

print("\nTop 10 Marken mit den MEISTEN Fehlern:")
print(f"{'Marke':<20} {'Total':<8} {'Errors':<8} {'Accuracy'}")
print("="*50)
for brand, stats in errors_sorted[:10]:
    print(f"{brand:<20} {stats['total']:<8} {stats['errors']:<8} {stats['accuracy']:.2f}%")

In [ ]:
# ===== 12. MODELL SPEICHERN =====
model_path = os.path.join(base_dir, r"Deep_learning_projet_SN_PP\final_model.h5")
model.save(model_path)
print(f"✓ Modell gespeichert: {model_path}")

# Speichern von Label Mappings
import json
mapping_path = os.path.join(base_dir, r"Deep_learning_projet_SN_PP\brand_mapping.json")
with open(mapping_path, 'w') as f:
    json.dump({'brand_to_idx': brand_to_idx, 'idx_to_brand': {str(k): v for k, v in idx_to_brand.items()}}, f)
print(f"✓ Label Mapping gespeichert: {mapping_path}")

In [ ]:
# ===== 13. PREDICTION AUF NEUEN BILDERN =====
# Funktion zum Vorhersagen auf beliebigen Bildern

def predict_car_brand(image_path, model, brand_to_idx):
    """Vorhersage für ein Bild"""
    from tensorflow.keras.preprocessing import image as img
    
    # Bild laden
    img_array = img.load_img(image_path, target_size=(224, 224))
    img_array = img.img_to_array(img_array) / 255.0
    img_array = np.expand_dims(img_array, axis=0)
    
    # Prediction
    prediction = model.predict(img_array, verbose=0)
    predicted_class = np.argmax(prediction[0])
    confidence = prediction[0][predicted_class] * 100
    
    return idx_to_brand[predicted_class], confidence

# Test auf ein paar Test-Bildern
print("\n===== BEISPIEL PREDICTIONS =====")
for idx in range(min(5, len(test_df))):
    image_path = test_df.iloc[idx]['path']
    true_brand = test_df.iloc[idx]['brand']
    pred_brand, confidence = predict_car_brand(image_path, model, brand_to_idx)
    match = "✓" if pred_brand == true_brand else "✗"
    print(f"{match} True: {true_brand:<15} | Predicted: {pred_brand:<15} | Confidence: {confidence:.2f}%")

In [4]:
import pandas as pd
from pathlib import Path
from PIL import Image

from sklearn import preprocessing
from sklearn.model_selection import train_test_split

import torch
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.transforms as transforms
from collections import Counter


#############################
# 1. Transforms definieren  #
#############################

train_transform = transforms.Compose([
    transforms.RandomAffine(
        degrees=20,
        translate=(0.2, 0.2),
        fill=(255, 255, 255)
    ),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=(0.8, 1.2)),
    transforms.ToTensor(),
])

val_transform = transforms.Compose([
    transforms.ToTensor(),  # meist keine Augmentation im Validation-Set
])


###################################
# 2. Dataset, das Bilder lazy lädt #
###################################

class CarBrandDataset(Dataset):
    def __init__(self, df, transform=None):
        """
        df: DataFrame mit mindestens Spalten ['brand', 'path', 'label']
        transform: torchvision.transforms
        """
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = Path(row["path"])  # in deiner CSV: absolute Windows-Pfade
        label = int(row["label"])

        # Bild laden
        image = Image.open(img_path).convert("RGB")

        # Augmentierungen anwenden
        if self.transform is not None:
            image = self.transform(image)

        return image, label


###############################################
# 3. CSV einlesen und Labels encodieren       #
#    (brand -> numerisches Label)             #
###############################################

csv_path = "combined_dataset.csv"  # ggf. anpassen
df = pd.read_csv(csv_path)

# Spalten erwartet: ['brand', 'image_file', 'path', 'source']
# -> wir encodieren 'brand' zu int-Labels
label_encoder = preprocessing.LabelEncoder()
df["label"] = label_encoder.fit_transform(df["brand"])


#########################################
# 4. Train/Val-Split (stratifiziert)    #
#########################################

train_idx, val_idx = train_test_split(
    df.index,
    test_size=0.2,
    random_state=42,
    stratify=df["label"],  # sorgt dafür, dass Klassenverteilung erhalten bleibt
)

train_df = df.loc[train_idx].reset_index(drop=True)
val_df = df.loc[val_idx].reset_index(drop=True)


#######################################################
# 5. Voll balancierte "intelligente Duplikation"      #
#    via WeightedRandomSampler                        #
#######################################################

# Häufigkeit pro Klasse im Trainings-Set
class_counts = train_df["label"].value_counts().sort_index()

# Ziel: jede Klasse soll so oft vorkommen wie die häufigste Klasse
max_count = class_counts.max()
n_classes = len(class_counts)

# Gesamtanzahl an Samples pro Epoche (voll balanciert)
num_samples_per_epoch = max_count * n_classes
print("Anzahl Klassen:", n_classes)
print("Max. Klassengröße:", max_count)
print("Samples pro Epoche (voll balanciert):", num_samples_per_epoch)

# Gewichte: 1 / Häufigkeit -> seltene Klassen haben höhere Gewichte
class_weights = 1.0 / class_counts

# Sample-Gewicht für jedes Bild anhand seiner Klasse
sample_weights = train_df["label"].map(class_weights).values
sample_weights = torch.DoubleTensor(sample_weights)

# WeightedRandomSampler mit Replacement
train_sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=num_samples_per_epoch,
    replacement=True  # wichtig für Oversampling
)


#########################################
# 6. Datasets und DataLoader bauen      #
#########################################

batch_size = 64  # anpassen nach RAM/GPU

train_dataset = CarBrandDataset(train_df, transform=train_transform)
val_dataset = CarBrandDataset(val_df, transform=val_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    sampler=train_sampler,  # -> nicht zusätzlich shuffle=True benutzen
    num_workers=4,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)


################################################
# 7. Optional: Klassenverteilung einer Epoche  #
################################################

def count_classes_in_loader(data_loader, label_encoder):
    counter = Counter()
    for images, labels in data_loader:
        counter.update(labels.tolist())

    print("\nVerteilung nach Sampler (eine Epoche):")
    for label_idx, count in sorted(counter.items()):
        brand_name = label_encoder.inverse_transform([label_idx])[0]
        print(f"{brand_name:20s} -> {count} Samples")

# EINMAL zum Debuggen aufrufen:
count_classes_in_loader(train_loader, label_encoder)

Anzahl Klassen: 35
Max. Klassengröße: 1113
Samples pro Epoche (voll balanciert): 38955


ValueError: num_samples should be a positive integer value, but got num_samples=38955

In [ ]:
import pandas as pd
from pathlib import Path
from PIL import Image

from sklearn import preprocessing
from sklearn.model_selection import train_test_split

import torch
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.transforms as transforms


#############################
# 1. Transforms definieren #
#############################

train_transform = transforms.Compose([
    transforms.RandomAffine(
        degrees=20,
        translate=(0.2, 0.2),
        fill=(255, 255, 255)
    ),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=(0.8, 1.2)),
    transforms.ToTensor(),
])

val_transform = transforms.Compose([
    transforms.ToTensor(),  # meist keine Augmentation im Validation-Set
])


###################################
# 2. Dataset, das Bilder lazy lädt #
###################################

class CarBrandDataset(Dataset):
    def __init__(self, df, transform=None):
        """
        df: DataFrame mit mindestens Spalten ['brand', 'path', 'label']
        transform: torchvision.transforms
        """
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = Path(row["path"])  # in deiner CSV: absolute Windows-Pfade
        label = int(row["label"])

        # Bild laden
        image = Image.open(img_path).convert("RGB")

        # Augmentierungen anwenden
        if self.transform is not None:
            image = self.transform(image)

        return image, label


###############################################
# 3. CSV einlesen und Labels encodieren      #
#    (brand -> numerisches Label)            #
###############################################

csv_path = "combined_dataset.csv"  # ggf. anpassen
df = pd.read_csv(csv_path)

# z.B. Spalten: ['brand', 'image_file', 'path', 'source']
# -> wir encodieren 'brand' zu int-Labels
label_encoder = preprocessing.LabelEncoder()
df["label"] = label_encoder.fit_transform(df["brand"])


#########################################
# 4. Train/Val-Split (stratifiziert)    #
#########################################

train_idx, val_idx = train_test_split(
    df.index,
    #test_size=0.2,
    random_state=42,
    stratify=df["label"],  # sorgt dafür, dass Klassenverteilung erhalten bleibt
)

train_df = df.loc[train_idx].reset_index(drop=True)
val_df = df.loc[val_idx].reset_index(drop=True)


############################################
# 5. "Intelligente Duplikation" via        #
#    WeightedRandomSampler                 #
############################################

# Häufigkeit pro Klasse
class_counts = train_df["label"].value_counts().sort_index()
# Gewicht = 1 / Häufigkeit -> seltene Klassen bekommen höhere Gewichte
class_weights = 1.0 / class_counts

# Sample-Gewicht für jedes Bild anhand seiner Klasse
sample_weights = train_df["label"].map(class_weights).values
sample_weights = torch.DoubleTensor(sample_weights)

# Anzahl Trainings-Samples pro Epoche:
# - Du kannst hier z.B. len(train_df) lassen
# - oder auch ein Vielfaches, um mehr Augmentation/„Duplikation“ zu erzwingen
num_samples_per_epoch = len(train_df)

train_sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=num_samples_per_epoch,
    replacement=True  # wichtig für Oversampling
)


#########################################
# 6. Datasets und DataLoader bauen      #
#########################################

batch_size = 64  # anpassen nach RAM/GPU

train_dataset = CarBrandDataset(train_df, transform=train_transform)
val_dataset = CarBrandDataset(val_df, transform=val_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    sampler=train_sampler,  # -> nicht shuffle gleichzeitig benutzen
    num_workers=4,          # je nach System
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)


#########################################
# 7. Beispiel: eine Batch auslesen      #
#########################################

images, labels = next(iter(train_loader))
print("Batch images shape:", images.shape)
print("Batch labels shape:", labels.shape)

C:\Users\pparr\AppData\Local\Temp\ipykernel_25228\758120413.py:105: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:219.)
  sample_weights = torch.DoubleTensor(sample_weights)
